# V6.5 — Local GPU YOLO11n@640 + Camber Checkpoints

**Bước này mới bắt đầu train.**

Đây chưa phải promotion/final V6.5. Mục tiêu là kiểm tra xem **nguồn dữ liệu mới có thật sự giúp đúng class đang yếu hay không** trước khi tốn compute cho YOLO11m@1024 và RT-DETR.

## Experiment

**Base:** V6.2E immutable  
**+ V6.5 candidate overlay:**
- MPCD train → `broken_strand`
- Power Equipment train → `broken_strand / foreign_object / bird_nest`
- relation-aware synthetic foreign objects → TRAIN only

**VAL:** V6.2E frozen VAL, không thay đổi  
**TEST:** không chạy

**Model:** YOLO11n  
**Resolution:** 640  
**Epochs:** 50  
**Purpose:** same-scale data ablation against old V6.2E YOLO11n@640 baseline.

## Vì sao chưa lấy V6.4 exact train làm base?

Artifact V6.4 hiện không replay được chính xác toàn bộ training runtime (79 GT edits + 600 hard crops). Vì vậy experiment này chỉ trả lời:

> "Các nguồn V6.5 mới có giá trị không?"

Nó **không** được gọi là final V6.5-vs-V6.4 comparison.

Nếu data ablation thắng, bước sau mới freeze canonical V6.5 và chạy YOLO11m@1024 / RT-DETR trên cùng canonical dataset.
## Cách chạy trên WSL

Notebook train trực tiếp trên GPU local. Camber Stash chỉ lưu dataset/artifact và kết quả; không submit job compute.

Chạy các cell cấu hình, kiểm tra GPU và chuẩn bị dataset trước. Cell **E. TRAIN** bắt đầu train 50 epoch, hiện log ngay trong notebook. Checkpoint được lưu local mỗi 5 epoch và có `best.pt` / `last.pt`. Sau cell **F. STRICT VAL**, chạy cell **G. UPLOAD** để lưu các checkpoint và báo cáo lên Camber. Upload diễn ra ở cell G, không tự đồng bộ trong lúc train.

Dữ liệu cache được giữ lại. Mỗi lần chạy cell CONFIG tạo một RUN_ID mới; nếu chỉ upload lỗi, chạy lại cell G, không chạy lại CONFIG hoặc TRAIN.

In [1]:
# ==============================================================================
# 0. LOCAL GPU CONFIG + CAMBER STASH AUTH
# ==============================================================================
from pathlib import Path
import os, sys, re, json, shutil, subprocess, hashlib, tarfile, zipfile, io
from datetime import datetime, timezone
from uuid import uuid4

STAGE_ROOT = "stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage"
COMPOSE_ROOT = f"{STAGE_ROOT}/compose_qa"
TRAIN_ROOT = "stash://bunpmc/projects/yolov11sdi/checkpoints/v6p5_data_ablation"

# Leave None to auto-discover when only one compose candidate exists.
COMPOSE_TAG_OVERRIDE = "v6p5_compose_4b9a452e24e2"

UPLOAD_CAMBER = True  # Upload results in the final cell; no remote compute job.

# Same YOLO11n@640 baseline scale
MODEL = "yolo11n.pt"
IMGSZ = 640
EPOCHS = 50
BATCH = 8  # YOLO11n@640; lower to 4 if local VRAM is insufficient.
DEVICE = 0
VAL_BATCH = 8
WORKERS = 2  # 4 workers exhausted WSL RAM -> cv2.imread returned None.
SEED = 20261007

key = os.environ.get("CAMBER_API_KEY")
if not key:
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret("CAMBER_API_KEY")
    except Exception:
        key = None

if not key:
    raise RuntimeError("Đặt CAMBER_API_KEY trong environment của kernel WSL (hoặc Kaggle Secrets).")

os.environ["CAMBER_API_KEY"] = key

camber_bin = Path.home() / ".camber/bin/camber"
if shutil.which("camber") is None and not camber_bin.exists():
    subprocess.check_call(
        "curl -sL https://cli.cambercloud.com/install-v2.sh | bash",
        shell=True,
    )

os.environ["PATH"] = (
    f"{Path.home()}/.camber/bin:{Path.home()}/.local/bin:"
    + os.environ.get("PATH","")
)
CAMBER = shutil.which("camber") or str(camber_bin)

assert Path(CAMBER).exists()

PROJECT = Path("/mnt/c/Users/intern/Documents/SEP490/train")
RAM_TMP = PROJECT / "artifacts" / "v6p5_train_local" / "metadata"
RAM_TMP.mkdir(parents=True, exist_ok=True)

print("CAMBER:", CAMBER)
print("COMPOSE_ROOT:", COMPOSE_ROOT)
# A new run directory prevents mixing datasets/checkpoints between experiments.
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
ROOT = PROJECT / "artifacts" / "v6p5_train_local" / RUN_ID
CACHE = PROJECT / "data" / "cache"
DATASET = ROOT / "dataset"
RUNS = ROOT / "runs"
for folder in [ROOT, CACHE, DATASET, RUNS]:
    folder.mkdir(parents=True, exist_ok=True)
print("Local run:", ROOT)
print("Train device:", DEVICE, "| Checkpoint destination:", TRAIN_ROOT)

CAMBER: /home/intern/.camber/bin/camber
COMPOSE_ROOT: stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage/compose_qa
Local run: /mnt/c/Users/intern/Documents/SEP490/train/artifacts/v6p5_train_local/20261008T181605Z_6b0eac5f
Train device: 0 | Checkpoint destination: stash://bunpmc/projects/yolov11sdi/checkpoints/v6p5_data_ablation


In [2]:
# 1. LOCAL DEPENDENCIES + GPU CHECK (NO TRAINING)
import pandas as pd
import numpy as np
import yaml
import torch
import ultralytics
from ultralytics import YOLO

if not torch.cuda.is_available():
    raise RuntimeError("CUDA chưa khả dụng trong kernel WSL. Kiểm tra PyTorch/CUDA trước khi train.")
if not isinstance(DEVICE, int) or not 0 <= DEVICE < torch.cuda.device_count():
    raise ValueError("DEVICE phải là chỉ số GPU local hợp lệ, ví dụ 0.")
if ultralytics.__version__ != "8.4.172":
    raise RuntimeError('Notebook dùng ultralytics==8.4.172. Cài bằng %pip install ultralytics==8.4.172 rồi restart kernel.')
print("GPU:", torch.cuda.get_device_name(DEVICE))
print("PyTorch:", torch.__version__, "| Ultralytics:", ultralytics.__version__)

GPU: NVIDIA GeForce RTX 3060 Laptop GPU
PyTorch: 2.14.1+cu130 | Ultralytics: 8.4.172


In [3]:
# ==============================================================================
# 2. SELECT COMPOSE ARTIFACTS FROM STASH
# ==============================================================================
def stash_ls_recursive(remote):
    r = subprocess.run(
        [CAMBER, "stash", "ls", "-r", remote],
        text=True,
        capture_output=True,
        env=os.environ.copy(),
    )
    if r.returncode != 0:
        raise RuntimeError(r.stderr or r.stdout)
    return r.stdout

if COMPOSE_TAG_OVERRIDE:
    COMPOSE_TAG = COMPOSE_TAG_OVERRIDE.strip()
else:
    listing = stash_ls_recursive(COMPOSE_ROOT)
    tags = sorted(set(re.findall(r"v6p5_compose_[0-9a-fA-F]{12}(?![0-9a-fA-F])", listing)))

    print("Compose candidates:", tags)

    if len(tags) == 0:
        raise RuntimeError(
            "Không tìm thấy compose candidate. "
            "Chạy KAGGLE_V6P5_CAMBER_COMPOSE_QA_LAUNCHER trước."
        )
    elif len(tags) == 1:
        COMPOSE_TAG = tags[0]
    else:
        raise RuntimeError(
            "Có nhiều compose candidate. "
            "Set COMPOSE_TAG_OVERRIDE ở cell CONFIG bằng tag muốn train: "
            + ", ".join(tags)
        )

if not re.fullmatch(r"v6p5_compose_[0-9a-fA-F]{12}", COMPOSE_TAG):
    raise ValueError(f"Invalid compose tag: {COMPOSE_TAG!r}")

print("SELECTED COMPOSE_TAG:", COMPOSE_TAG)
COMPOSE_REMOTE = f"{COMPOSE_ROOT}/{COMPOSE_TAG}"
# Read only: verify the selected index and all referenced archives before submission.
index_local = RAM_TMP / f"{COMPOSE_TAG}.index.json"
r = subprocess.run(
    [CAMBER, "stash", "cp", f"{COMPOSE_REMOTE}/{COMPOSE_TAG}.index.json", str(index_local)],
    text=True, capture_output=True, env=os.environ.copy(),
)
if r.returncode != 0:
    raise RuntimeError(r.stderr or r.stdout)
compose_index = json.loads(index_local.read_text())
if compose_index.get("compose_tag") != COMPOSE_TAG:
    raise ValueError("Compose index tag does not match the selected tag.")
for remote in [
    f"{COMPOSE_REMOTE}/{COMPOSE_TAG}.freeze.json",
    compose_index["synthetic_archive"],
    compose_index["direct_archives"]["mpcd"],
    compose_index["direct_archives"]["power_equipment"],
]:
    stash_ls_recursive(remote)
    print("Artifact available:", remote)

SELECTED COMPOSE_TAG: v6p5_compose_4b9a452e24e2
Artifact available: stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage/compose_qa/v6p5_compose_4b9a452e24e2/v6p5_compose_4b9a452e24e2.freeze.json
Artifact available: stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage/compose_qa/v6p5_compose_4b9a452e24e2/v6p5_compose_4b9a452e24e2_synthetic_approved.tar.gz
Artifact available: stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage/mpcd/mpcd_broken_strand_stage.tar.gz
Artifact available: stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage/power_equipment/power_equipment_stage.tar.gz


In [4]:
# 3. DATASET HELPERS
V6ROOT = "stash://bunpmc/projects/yolov11sdi/datasets"
V6TAG = "v6p2e_23562a45b343"
COMPOSE_CACHE = CACHE / COMPOSE_TAG
COMPOSE_CACHE.mkdir(parents=True, exist_ok=True)
def run(cmd):
    r=subprocess.run(cmd,text=True,capture_output=True,env=os.environ.copy())
    if r.returncode!=0:
        raise RuntimeError((r.stderr or r.stdout)[:10000])
    return r

def get(remote,local):
    local=Path(local)
    local.parent.mkdir(parents=True,exist_ok=True)
    if not local.is_file() or local.stat().st_size == 0:
        partial=local.with_name(local.name+".part")
        run([CAMBER,"stash","cp",remote,str(partial)])
        if not partial.is_file() or partial.stat().st_size == 0:
            raise RuntimeError(f"Empty download: {remote}")
        partial.replace(local)
    return local

def put(local,remote):
    run([CAMBER,"stash","cp",str(local),remote])

def norm_names(names):
    if isinstance(names,dict):
        return [names[k] for k in sorted(names,key=lambda x:int(x))]
    return list(names or [])

def safe_link_or_copy(src,dst):
    src=Path(src); dst=Path(dst)
    dst.parent.mkdir(parents=True,exist_ok=True)
    if dst.exists():
        return
    try:
        os.link(src,dst)
    except Exception:
        shutil.copy2(src,dst)

def sha256_file(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        while True:
            b=f.read(1024*1024)
            if not b: break
            h.update(b)
    return h.hexdigest()

In [5]:
# =============================================================================
# A. RECONSTRUCT V6.2E BASE
# =============================================================================
index_local=get(f"{COMPOSE_REMOTE}/{COMPOSE_TAG}.index.json",COMPOSE_CACHE/"compose.index.json")
compose_index=json.loads(index_local.read_text())
if compose_index.get("compose_tag") != COMPOSE_TAG:
    raise RuntimeError("Compose index tag mismatch")
compose_freeze_path=get(f"{COMPOSE_REMOTE}/{COMPOSE_TAG}.freeze.json",COMPOSE_CACHE/"compose.freeze.json")
compose_freeze=json.loads(compose_freeze_path.read_text())
if compose_freeze.get("tag") != COMPOSE_TAG:
    raise RuntimeError("Compose freeze tag mismatch")

freeze_path=get(
    f"{V6ROOT}/v6p2e/{V6TAG}/v6p2e.freeze.json",
    CACHE/"v6p2e.freeze.json",
)
freeze=json.loads(freeze_path.read_text())

base_tar=get(
    f"{V6ROOT}/curated_v6_rs1280_{freeze['base_tag']}.tar",
    CACHE/f"base_{freeze['base_tag']}.tar",
)
overlay_tar=get(
    f"{V6ROOT}/v6p1/{freeze['parent_dataset_tag']}/v6p1_fotl_overlay_{freeze['overlay_tag']}.tar.gz",
    CACHE/f"overlay_{freeze['overlay_tag']}.tar.gz",
)
labels_zip=get(
    f"{V6ROOT}/v6p2e/{V6TAG}/v6p2e_reconciled_labels.zip",
    CACHE/"v6p2e_reconciled_labels.zip",
)
manifest_csv=get(
    f"{V6ROOT}/v6p2e/{V6TAG}/v6p2e_split_manifest.csv",
    CACHE/"v6p2e_split_manifest.csv",
)

BASE_EX=CACHE/"base_extract"
OV_EX=CACHE/"overlay_extract"
RECON=CACHE/"reconciled_labels"
for p in [BASE_EX,OV_EX,RECON]:
    p.mkdir(parents=True,exist_ok=True)

def find_sixclass_root(root):
    target=[
        "insulator",
        "insulator_broken",
        "insulator_flashover",
        "bird_nest",
        "foreign_object",
        "broken_strand",
    ]
    for yp in Path(root).rglob("data.yaml"):
        try:
            cfg=yaml.safe_load(yp.read_text())
            if norm_names(cfg.get("names"))==target:
                return yp.parent
        except Exception:
            pass
    raise RuntimeError(f"Six-class root not found under {root}")

if not any(BASE_EX.rglob("data.yaml")):
    with tarfile.open(base_tar,"r:*") as t:
        t.extractall(BASE_EX,filter="data")

if not any(OV_EX.rglob("data.yaml")):
    with tarfile.open(overlay_tar,"r:*") as t:
        t.extractall(OV_EX,filter="data")

if not any(RECON.rglob("*.txt")):
    with zipfile.ZipFile(labels_zip) as z:
        z.extractall(RECON)

BASE_ROOT=find_sixclass_root(BASE_EX)
OV_ROOT=find_sixclass_root(OV_EX)
manifest=pd.read_csv(manifest_csv)

def locate(row):
    old=str(row["image_path_source"]).replace("\\","/")
    oldsplit=str(row["previous_v6p1_split"])
    marker=f"/images/{oldsplit}/"
    suffix=old.split(marker,1)[1]

    if suffix.startswith("base/"):
        root=BASE_ROOT; rel=suffix[len("base/"):]
    elif suffix.startswith("fotl/"):
        root=OV_ROOT; rel=suffix[len("fotl/"):]
    else:
        raise RuntimeError(suffix)

    return (
        root/"images"/oldsplit/rel,
        root/"labels"/oldsplit/Path(rel).with_suffix(".txt"),
    )

def label_for(row,lp0):
    rp=RECON/f"{row['sha256']}.txt"
    if str(row.get("resolution","")).startswith("merged") and rp.exists():
        return rp
    return lp0

for s in ["train","val"]:
    (DATASET/"images"/s).mkdir(parents=True,exist_ok=True)
    (DATASET/"labels"/s).mkdir(parents=True,exist_ok=True)

unresolved=[]

for i,row in manifest[manifest["split"].isin(["train","val"])].iterrows():
    split=str(row["split"])
    ip,lp0=locate(row)
    lp=label_for(row,lp0)

    if not ip.exists() or not lp.exists():
        unresolved.append((int(i),str(ip),str(lp)))
        continue

    name=f"{str(row['sha256'])[:16]}_{i:06d}{ip.suffix.lower()}"
    safe_link_or_copy(ip,DATASET/"images"/split/name)
    safe_link_or_copy(lp,DATASET/"labels"/split/Path(name).with_suffix(".txt"))

if unresolved:
    pd.DataFrame(unresolved,columns=["row","image","label"]).to_csv(
        ROOT/"unresolved.csv",index=False
    )
    raise RuntimeError(f"V6.2E unresolved files: {len(unresolved)}")

base_train_count=len(list((DATASET/"images/train").glob("*")))
base_val_count=len(list((DATASET/"images/val").glob("*")))

In [6]:
# =============================================================================
# B. ADD DIRECT V6.5 TRAIN OVERLAYS
# =============================================================================
overlay_sources=[
    (
        compose_index["direct_archives"]["mpcd"],
        COMPOSE_CACHE/"mpcd.tar.gz",
        "mpcd_broken_strand",
    ),
    (
        compose_index["direct_archives"]["power_equipment"],
        COMPOSE_CACHE/"pe.tar.gz",
        "power_equipment_stage",
    ),
]

source_stage_freeze = compose_freeze["source_stage_freeze"]
source_metadata = source_stage_freeze.get("sources", {})
expected_direct = source_stage_freeze.get("counts", {}).get("direct_train_images")
direct_source_summary = []
added_direct=0

for remote,local,rootname in overlay_sources:
    source_key = "MPCD" if rootname == "mpcd_broken_strand" else "PowerEquipment"
    availability = source_metadata.get(source_key, {}).get("availability", {})
    if availability.get("used_for_training") is False:
        # Do not consume mutable shared archives for sources excluded by this freeze.
        prefix = "mpcd" if source_key == "MPCD" else "pe"
        if any((DATASET / "images/train").glob(f"v65_{prefix}_*")):
            raise RuntimeError(f"Existing {source_key} images conflict with selected freeze. Start a new CONFIG run.")
        print(f"SKIP {source_key}: {availability.get('status', 'not used for training')}")
        print("Reason:", availability.get("reason", "Excluded by stage metadata"))
        direct_source_summary.append({"source": source_key, "train_images": 0, "availability": availability})
        continue
    get(remote,local)

    source_added=0
    with tarfile.open(local,"r:*") as t:
        # Stream only train images/labels to avoid materializing external_eval.
        members={m.name:m for m in t.getmembers() if m.isfile()}

        image_members=[
            n for n in members
            if f"{rootname}/images/train/" in n
        ]

        for n in image_members:
            m=members[n]
            raw=t.extractfile(m).read()

            basename=Path(n).name
            source_prefix="mpcd" if "mpcd" in rootname else "pe"
            name=f"v65_{source_prefix}_{basename}"

            di=DATASET/"images/train"/name
            di.write_bytes(raw)

            label_name=str(
                Path(n.replace("/images/train/","/labels/train/"))
                .with_suffix(".txt")
            )

            lm=members.get(label_name)
            if lm is None:
                raise RuntimeError(f"Missing overlay label: {label_name}")

            dl=DATASET/"labels/train"/Path(name).with_suffix(".txt")
            dl.write_bytes(t.extractfile(lm).read())
            added_direct+=1
            source_added+=1

    if source_added == 0:
        raise RuntimeError(f"No TRAIN images found in direct overlay: {remote}")
    direct_source_summary.append({"source": source_key, "train_images": source_added, "availability": availability})
    print(rootname, "TRAIN images:", source_added)
    # Retain downloaded source archive for subsequent local runs.
if expected_direct is not None and added_direct != int(expected_direct):
    raise RuntimeError(f"Direct TRAIN count differs from stage freeze: {added_direct} != {expected_direct}")
print("Total direct TRAIN additions:", added_direct)
if added_direct == 0:
    print("Experiment: V6.2E + approved synthetic only; no new MPCD/PowerEquipment training images.")


SKIP MPCD: reference_only
Reason: RuntimeError('MPCD Google Drive download unavailable: FileURLRetrievalError("Failed to retrieve file url:\\n\\n\\tCannot retrieve the public link of the file. You may need to change\\n\\tthe permission to \'Anyone with the link\', or have had many accesses.\\n\\tCheck FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.\\n\\nYou may still be able to access the file from the browser:\\n\\n\\thttps://drive.google.com/uc?id=1KyciMmwL2_p_-mwckHFqG5fkBA1jzATv\\n\\nbut Gdown can\'t. Please check connections and permissions.")')
SKIP PowerEquipment: reference_only
Reason: ZeroDivisionError('float division by zero')
Total direct TRAIN additions: 0
Experiment: V6.2E + approved synthetic only; no new MPCD/PowerEquipment training images.


In [7]:
# =============================================================================
# C. ADD SYNTHETIC V6.5 TRAIN OVERLAY
# =============================================================================
# Discover exact synthetic archive filename from compose index.
index_remote=f"{COMPOSE_REMOTE}/{COMPOSE_TAG}.index.json"
index_local=get(index_remote,COMPOSE_CACHE/"compose.index.json")
compose_index=json.loads(index_local.read_text())

synthetic_remote=compose_index["synthetic_archive"]
synthetic_local=get(synthetic_remote,COMPOSE_CACHE/"synthetic.tar.gz")

expected_synthetic_sha=compose_freeze["artifacts"]["synthetic_archive"]["sha256"]
if sha256_file(synthetic_local) != expected_synthetic_sha:
    raise RuntimeError("Synthetic archive SHA256 mismatch")

added_synth=0

with tarfile.open(synthetic_local,"r:gz") as t:
    members={m.name:m for m in t.getmembers() if m.isfile()}

    image_members=[
        n for n in members
        if "synthetic_foreign/images/train/" in n
    ]

    for n in image_members:
        m=members[n]
        basename=Path(n).name
        name=f"v65_syn_{basename}"

        (DATASET/"images/train"/name).write_bytes(
            t.extractfile(m).read()
        )

        label_name=str(
            Path(n.replace("/images/train/","/labels/train/"))
            .with_suffix(".txt")
        )

        lm=members.get(label_name)
        if lm is None:
            raise RuntimeError(f"Missing synthetic label: {label_name}")

        (DATASET/"labels/train"/Path(name).with_suffix(".txt")).write_bytes(
            t.extractfile(lm).read()
        )

        added_synth+=1

if added_synth != int(compose_freeze["counts"]["approved_synthetic"]):
    raise RuntimeError("Synthetic image count does not match compose freeze")
# Retain the verified synthetic archive in the compose-specific cache.

In [8]:
# =============================================================================
# D. WRITE DATA YAML
# =============================================================================
names=[
    "insulator",
    "insulator_broken",
    "insulator_flashover",
    "bird_nest",
    "foreign_object",
    "broken_strand",
]

data_yaml=DATASET/"data.yaml"
data_yaml.write_text(yaml.safe_dump({
    "path":str(DATASET),
    "train":"images/train",
    "val":"images/val",
    "names":{i:n for i,n in enumerate(names)},
},sort_keys=False))

train_count=len(list((DATASET/"images/train").glob("*")))
val_count=len(list((DATASET/"images/val").glob("*")))


expected_train = int((manifest["split"] == "train").sum()) + added_direct + added_synth
expected_val = int((manifest["split"] == "val").sum())
if train_count != expected_train or val_count != expected_val:
    raise RuntimeError(f"Dataset count mismatch: train={train_count}/{expected_train}, val={val_count}/{expected_val}")
print("Local dataset:", data_yaml)
print("TRAIN:", train_count, "| frozen VAL:", val_count)
print("Direct additions:", added_direct, "| synthetic additions:", added_synth)

Local dataset: /mnt/c/Users/intern/Documents/SEP490/train/artifacts/v6p5_train_local/20261008T181605Z_6b0eac5f/dataset/data.yaml
TRAIN: 17983 | frozen VAL: 2272
Direct additions: 0 | synthetic additions: 500


In [9]:
# =============================================================================
# E. TRAIN SAME-SCALE YOLO11n@640 / 50 epochs
# =============================================================================
# Train from a WSL ext4 copy: /mnt/c reads are slow and fail intermittently under DataLoader workers.
FAST_DATASET = Path.home() / "datasets" / COMPOSE_TAG
FAST_DATASET.mkdir(parents=True, exist_ok=True)
subprocess.run(["rsync", "-a", "--delete", f"{DATASET}/", f"{FAST_DATASET}/"], check=True)
data_yaml = FAST_DATASET / "data.yaml"
data_yaml.write_text(yaml.safe_dump({
    "path": str(FAST_DATASET),
    "train": "images/train",
    "val": "images/val",
    "names": {i: n for i, n in enumerate(names)},
}, sort_keys=False))
print("Train dataset:", data_yaml)

model=YOLO(MODEL)

result=model.train(
    data=str(data_yaml),
    imgsz=IMGSZ,
    epochs=EPOCHS,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    cache=False,
    save=True,
    save_period=5,
    pretrained=True,
    seed=SEED,
    project=str(RUNS),
    name="yolo11n_v6p5_data_ablation_640",
    exist_ok=True,
    plots=False,
    verbose=True,
)

run_dir=Path(model.trainer.save_dir)
best=run_dir/"weights/best.pt"
assert best.exists(), best


print("Local checkpoint directory:", run_dir / "weights")
print("Best checkpoint:", best)
print("Last checkpoint:", run_dir / "weights" / "last.pt")

Train dataset: /home/intern/datasets/v6p5_compose_4b9a452e24e2/data.yaml
New https://pypi.org/project/ultralytics/8.4.174 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.172 🚀 Python-3.14.4 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/intern/datasets/v6p5_compose_4b9a452e24e2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7,

/mnt/c/Users/intern/Documents/SEP490/train/.venv/lib/python3.14/site-packages/triton/backends/nvidia/driver.c:9:10: fatal error: Python.h: No such file or directory
    9 | #include <Python.h>
      |          ^~~~~~~~~~
compilation terminated.


YOLO11n summary (fused): 100 layers, 2,583,322 parameters, 0 gradients, 6.4 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 142/142 6.2it/s 22.8s0.1s
                   all       2272       2903      0.805      0.694      0.769      0.418
             insulator        322        778      0.892       0.93      0.948      0.652
      insulator_broken        656        791      0.843      0.694      0.789      0.405
   insulator_flashover        140        163      0.845      0.626      0.751      0.307
             bird_nest        717        821      0.839        0.8      0.833      0.473
        foreign_object         71         98      0.623      0.455      0.556      0.319
         broken_strand        248        252      0.788      0.659      0.737      0.352
Speed: 0.2ms preprocess, 2.7ms inference, 0.0ms loss, 2.0ms postprocess per image
Local checkpoint directory: /mnt/c/Users/intern/Documents/SEP490/train/artifac

In [10]:
# =============================================================================
# F. STRICT VAL EVALUATION
# =============================================================================
# Release the training model's GPU memory before loading the best checkpoint.
import gc
model.cpu()
del model
gc.collect()
torch.cuda.empty_cache()
best_model=YOLO(str(best))

metrics=best_model.val(
    data=str(data_yaml),
    split="val",
    imgsz=IMGSZ,
    batch=VAL_BATCH,
    device=DEVICE,
    plots=False,
    verbose=False,
)

m=metrics.box
all_ap=np.asarray(m.all_ap,dtype=float)
ap50=np.asarray(m.ap50,dtype=float)
class_ids=[int(x) for x in np.asarray(m.ap_class_index).tolist()]

if all_ap.ndim==1:
    all_ap=all_ap[:,None]

thresholds=0.50+0.05*np.arange(all_ap.shape[1])

def col_for(t):
    return int(np.argmin(np.abs(thresholds-t)))

per_class=[]

for ri,cid in enumerate(class_ids):
    vals=all_ap[ri]
    row={
        "cid":cid,
        "class":names[cid],
        "AP50":float(ap50[ri]),
        "AP50_95":float(np.mean(vals)),
        "gap":float(ap50[ri]-np.mean(vals)),
        "AP75":float(vals[col_for(0.75)]) if all_ap.shape[1]>1 else None,
        "AP90":float(vals[col_for(0.90)]) if all_ap.shape[1]>1 else None,
        "AP95":float(vals[col_for(0.95)]) if all_ap.shape[1]>1 else None,
    }
    per_class.append(row)

# Old V6.2E YOLO11n@640 reference
baseline={
    "mAP50":0.768,
    "mAP50_95":0.413,
    "gap":0.355,
    "per_class":{
        "insulator":{"AP50":0.934,"AP50_95":0.638},
        "insulator_broken":{"AP50":0.800,"AP50_95":0.407},
        "insulator_flashover":{"AP50":0.755,"AP50_95":0.326},
        "bird_nest":{"AP50":0.831,"AP50_95":0.455},
        "foreign_object":{"AP50":0.537,"AP50_95":0.295},
        "broken_strand":{"AP50":0.749,"AP50_95":0.356},
    },
}

report={
    "experiment":("V6.2E + V6.5 synthetic-only data ablation" if added_direct == 0
                  else "V6.2E + V6.5 external candidate data ablation"),
    "direct_sources":direct_source_summary,
    "compose_tag":COMPOSE_TAG,
    "run_id":RUN_ID,
    "seed":SEED,
    "batch":BATCH,
    "run_dir":str(run_dir),
    "model":"YOLO11n",
    "imgsz":IMGSZ,
    "epochs":EPOCHS,
    "base_train_images":base_train_count,
    "added_direct_train_images":added_direct,
    "added_synthetic_train_images":added_synth,
    "final_train_images":train_count,
    "val_images":val_count,
    "mAP50":float(m.map50),
    "mAP50_95":float(m.map),
    "gap":float(m.map50-m.map),
    "precision":float(m.mp),
    "recall":float(m.mr),
    "per_class":per_class,
    "reference_v6p2e_yolo11n":baseline,
}

report["delta_vs_v6p2e"]={
    "mAP50":report["mAP50"]-baseline["mAP50"],
    "mAP50_95":report["mAP50_95"]-baseline["mAP50_95"],
    "gap":report["gap"]-baseline["gap"],
}

for row in report["per_class"]:
    b=baseline["per_class"].get(row["class"])
    if b:
        row["delta_AP50_vs_v6p2e"]=row["AP50"]-b["AP50"]
        row["delta_AP50_95_vs_v6p2e"]=row["AP50_95"]-b["AP50_95"]

report_path=ROOT/"v6p5_yolo11n_data_ablation_report.json"
report_path.write_text(json.dumps(report,indent=2))

pd.DataFrame(per_class).to_csv(
    ROOT/"v6p5_yolo11n_strict_iou_per_class.csv",
    index=False,
)

# Practical promotion signal for the NEW sources:
# foreign AP50/AP50:95 should improve, strand strict metric should improve,
# and global AP50:95 must not materially regress.
pc={r["class"]:r for r in per_class}

promotion={
    "global_map5095_not_regress_gt_1pt":
        report["mAP50_95"] >= baseline["mAP50_95"]-0.01,
    "foreign_AP50_improves_2pt":
        pc.get("foreign_object",{}).get("AP50",0) >= baseline["per_class"]["foreign_object"]["AP50"]+0.02,
    "foreign_AP5095_improves_1pt":
        pc.get("foreign_object",{}).get("AP50_95",0) >= baseline["per_class"]["foreign_object"]["AP50_95"]+0.01,
    "strand_AP5095_improves_1pt":
        pc.get("broken_strand",{}).get("AP50_95",0) >= baseline["per_class"]["broken_strand"]["AP50_95"]+0.01,
}

promotion["candidate_pass"]=all(promotion.values())
report["promotion_gate"]=promotion
report_path.write_text(json.dumps(report,indent=2))

Ultralytics 8.4.172 🚀 Python-3.14.4 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
YOLO11n summary (fused): 100 layers, 2,583,322 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2799.2±1302.1 MB/s, size: 316.6 KB)
val: Scanning /home/intern/datasets/v6p5_compose_4b9a452e24e2/labels/val.cache... 2272 images, 179 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2272/2272 595.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 284/284 1.4it/s 3:18<0.1ss0
                   all       2272       2903      0.807      0.694      0.769      0.417
Speed: 0.7ms preprocess, 4.0ms inference, 0.0ms loss, 1.7ms postprocess per image


5133

In [11]:
# G. UPLOAD LOCAL CHECKPOINTS + REPORTS TO CAMBER STASH
# Run after training/evaluation. Rerun this cell alone if an upload fails.
# All local data and checkpoints are retained.
tag_input=json.dumps({
    "compose_tag": COMPOSE_TAG, "model": MODEL,
    "imgsz": IMGSZ, "epochs": EPOCHS, "batch": BATCH, "seed": SEED,
    "run_id": RUN_ID,
}, sort_keys=True).encode()
tag="v6p5_y11n_ablation_"+hashlib.sha256(tag_input).hexdigest()[:12]
remote_root=f"{TRAIN_ROOT}/{tag}"

if UPLOAD_CAMBER:
    uploads = [(best, f"{remote_root}/best.pt"),
               (run_dir / "weights" / "last.pt", f"{remote_root}/last.pt"),
               (report_path, f"{remote_root}/{report_path.name}"),
               (ROOT / "v6p5_yolo11n_strict_iou_per_class.csv",
                f"{remote_root}/v6p5_yolo11n_strict_iou_per_class.csv")]
    for name in ["results.csv", "args.yaml"]:
        if (run_dir / name).is_file():
            uploads.append((run_dir / name, f"{remote_root}/{name}"))
    for checkpoint in sorted((run_dir / "weights").glob("epoch*.pt")):
        uploads.append((checkpoint, f"{remote_root}/weights/{checkpoint.name}"))
    for local, remote in uploads:
        if not local.is_file():
            raise FileNotFoundError(local)
        print("UPLOAD:", local.name)
        put(local, remote)
    done=ROOT / "done.json"
    done.write_text(json.dumps({
        "tag": tag, "remote_root": remote_root, "local_run_dir": str(run_dir),
        "promotion_gate": promotion,
    }, indent=2))
    put(done, f"{remote_root}/done.json")
    print("Upload complete:", remote_root)
else:
    print("UPLOAD_CAMBER=False: checkpoints/reports retained locally.")
print("Local checkpoints:", run_dir / "weights")

UPLOAD: best.pt
UPLOAD: last.pt
UPLOAD: v6p5_yolo11n_data_ablation_report.json
UPLOAD: v6p5_yolo11n_strict_iou_per_class.csv
UPLOAD: results.csv
UPLOAD: args.yaml
UPLOAD: epoch0.pt
UPLOAD: epoch10.pt
UPLOAD: epoch15.pt
UPLOAD: epoch20.pt
UPLOAD: epoch25.pt
UPLOAD: epoch30.pt
UPLOAD: epoch35.pt
UPLOAD: epoch40.pt
UPLOAD: epoch45.pt
UPLOAD: epoch5.pt
Upload complete: stash://bunpmc/projects/yolov11sdi/checkpoints/v6p5_data_ablation/v6p5_y11n_ablation_761a2f41098d
Local checkpoints: /mnt/c/Users/intern/Documents/SEP490/train/artifacts/v6p5_train_local/20261008T181605Z_6b0eac5f/runs/yolo11n_v6p5_data_ablation_640/weights


## Cách đọc kết quả

Đây là **data ablation**, nên compare với YOLO11n@640 V6.2E cũ:

- global `mAP50`: 0.768
- global `mAP50:95`: 0.413
- gap: 0.355

Đặc biệt:

- `foreign_object`: 0.537 / 0.295
- `broken_strand`: 0.749 / 0.356

### Gate để đi tiếp YOLO11m@1024

Candidate được xem là đáng giữ nếu:

- global mAP50:95 không giảm quá 1 điểm;
- foreign AP50 tăng ≥ 2 điểm;
- foreign AP50:95 tăng ≥ 1 điểm;
- broken_strand AP50:95 tăng ≥ 1 điểm.

Nếu gate pass:

`review/promotion -> canonical V6.5 -> YOLO11m@1024 40ep -> RT-DETR B0`

Nếu foreign tăng nhưng strand giảm, tách ablation nguồn:
- only foreign expansion
- only broken-strand direct sources

Không cần cưới cả hai source chỉ vì chúng cùng xuất hiện trong một notebook. Dữ liệu cũng nên có quyền chia tay.